# grid-rbd JAX backend — jit, vmap & differentiation

The **JAX backend** wraps the same compiled per-robot `.so` behind
[`jax.ffi`](https://docs.jax.dev/en/latest/ffi.html) targets, so GRiD's
dynamics/kinematics/gradient kernels are callable inside `jax.jit` and run
device-resident on JAX-managed CUDA streams. Register with
`grid_rbd.jax.register_robot(...)` to get a `JaxRobotHandle` whose methods
return `jax.Array`s.

This is the JAX parallel to notebook **02** (torch). We show the JAX-specific
value — **`jax.jit`** and **`jax.vmap`** — plus how differentiation works on
this backend (see section 4: the FFI calls are not auto-differentiable, so the
derivatives come from GRiD's own **analytic** gradient kernels, which we
cross-check by finite difference).

**Setup:** a CUDA GPU + `nvcc` on PATH, and the JAX extra installed editable
from this repo — `pip install -e "python/[jax]"` from the repo root (see
[`notebooks/README.md`](README.md)). iiwa14 caches in seconds; the first
`jax.jit` trace adds a little XLA warm-up.

In [ ]:
import numpy as np
import jax, jax.numpy as jnp
from pathlib import Path
import grid_rbd            # numpy handle, for the cross-check oracle
import grid_rbd.jax as grid_jax

URDF = next(p / 'robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'robot_assets' / 'iiwa14.urdf').exists())
assert URDF.exists(), URDF
np.random.seed(0)
print('jax', jax.__version__, ' devices:', jax.devices())

## 1. Register with the JAX backend

`grid_jax.register_robot` compiles/caches the **same** `.so` as the plain
`grid_rbd.register_robot` (cache hit if already built) and additionally
registers the JAX FFI targets. We also grab a plain numpy handle on the same
robot to use as a validation oracle.

In [ ]:
h = grid_jax.register_robot('iiwa14_jax_nb', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64)
h_np = grid_rbd.register_robot('iiwa14_jax_nb_oracle', urdf_path=str(URDF),
                               floating_base=False, max_batch_size=64)
NJ, NV = h.num_joints, h.num_vel
print(h.__class__.__name__, ' NJ =', NJ, ' NV =', NV)

## 2. `jax.jit` a dynamics call

The handle methods are `jax.jit`-traceable: the FFI target executes inside the
compiled XLA program. We jit `forward_dynamics` and validate against the numpy
handle (same `.so`, so they agree to float32 round-off).

In [ ]:
B = 8
q  = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)
qd = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)
u  = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)

@jax.jit
def fd(q, qd, u):
    return h.forward_dynamics(q, qd, u)

qdd = fd(q, qd, u)                 # traced + compiled once, then cached
qdd_np = h_np.forward_dynamics(np.asarray(q), np.asarray(qd), np.asarray(u))
err = float(jnp.max(jnp.abs(qdd - jnp.asarray(qdd_np))))
print('jit forward_dynamics:', qdd.shape, '  max|jax - numpy| =', f'{err:.2e}')
assert qdd.shape == (B, NJ)
assert err < 1e-4, err

## 3. `jax.vmap` over a batch

The kernels already batch over axis 0, but `jax.vmap` lets you write the
per-sample computation and map it — the idiomatic JAX way to compose a
single-sample function over a batch. We map a single-state `forward_dynamics`
and check it matches the natively-batched call.

In [ ]:
# The FFI targets are registered with a fixed batch dimension, so JAX's
# batching (vmap) transform isn't supported directly on a single-sample call.
# The kernels are *already* batched over axis 0, so you batch by passing a
# stacked (B, NJ) array — the GRiD-native equivalent of vmap. (To use vmap on
# top, wrap the call in jax.pure_callback or batch manually.)
try:
    def fd_one(qi, qdi, ui):
        return h.forward_dynamics(qi[None], qdi[None], ui[None])[0]
    qdd_vmap = jax.vmap(fd_one)(q, qd, u)
    print('vmap supported:', qdd_vmap.shape)
except Exception as e:
    print('vmap over the FFI call is not supported on this backend:',
          type(e).__name__)
    print('use the native batch axis instead — h.forward_dynamics((B,NJ) array):')
    qdd_batched = h.forward_dynamics(q, qd, u)   # already batched over axis 0
    print('  native-batched forward_dynamics:', qdd_batched.shape)
    assert qdd_batched.shape == (B, NJ)

## 4. Differentiation — analytic gradient kernels (not autodiff)

The GRiD FFI calls are **opaque** to JAX's autodiff (no JVP/VJP rule is
registered for the foreign call), so `jax.grad` / `jax.jacobian` **through**
`forward_dynamics` is not available on this backend — that's the torch
backend's job (notebook 02). Instead, GRiD exposes its derivatives as
**first-class analytic kernels**: `forward_dynamics_gradient` /
`inverse_dynamics_gradient` (first order) and `idsva_so` / `fdsva_so` (second
order), all `jax.jit`-callable. Below we jit the analytic Jacobian and validate
it against a finite-difference of the forward map — and confirm the autodiff
path is indeed unavailable.

In [ ]:
# Confirm jax.grad through the FFI call is NOT differentiable (a backend limit).
def scalar(qq):
    return jnp.sum(h.forward_dynamics(qq[None], qd[:1], u[:1]) ** 2)
try:
    jax.grad(scalar)(q[0])
    grad_through_ffi = True
except Exception as e:
    grad_through_ffi = False
    print('as expected, jax.grad through forward_dynamics is unsupported:',
          type(e).__name__)
assert not grad_through_ffi, 'FFI unexpectedly became differentiable — update this note'

In [ ]:
# The supported path: GRiD's analytic forward-dynamics gradient, jitted.
q1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
qd1 = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
u1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)

fd_grad = jax.jit(h.forward_dynamics_gradient)
G = fd_grad(q1, qd1, u1)               # (1, NJ, 2*NJ) = [dqdd/dq | dqdd/dqd]
df_dq, df_dqd = G[..., :NJ], G[..., NJ:]
print('forward_dynamics_gradient:', G.shape)

# Central finite-difference of forward_dynamics for the cross-check.
def fd_jac(fn, x, eps=1e-3):
    n = x.shape[1]
    J = np.zeros((NJ, n))
    for j in range(n):
        dx = np.zeros_like(x); dx[0, j] = eps
        J[:, j] = (np.asarray(fn(x + dx))[0] - np.asarray(fn(x - dx))[0]) / (2 * eps)
    return J

fdq  = fd_jac(lambda qq: h.forward_dynamics(qq, qd1, u1), q1)
fdqd = fd_jac(lambda vv: h.forward_dynamics(q1, vv, u1), qd1)
e_dq  = float(np.max(np.abs(np.asarray(df_dq[0]) - fdq)))
e_dqd = float(np.max(np.abs(np.asarray(df_dqd[0]) - fdqd)))
print(f'analytic dqdd/dq  vs FD: max|err| = {e_dq:.2e}')
print(f'analytic dqdd/dqd vs FD: max|err| = {e_dqd:.2e}')
assert e_dq < 5e-2 and e_dqd < 5e-2, (e_dq, e_dqd)

## 5. Second-order `idsva_so` vs the numpy oracle

The second-order kernel is on the JAX surface too. `idsva_so(q, qd, qdd)`
returns four `(B, NV, NV, NV)` tensors `(d2tau_dq, d2tau_dqd, d2tau_cross,
dM_dq)`; we jit it and validate block-for-block against the numpy handle
(same `.so`, so they agree to float32 round-off).

In [ ]:
qdd1 = jnp.zeros((1, NJ), dtype=jnp.float32)
idsva_jax = jax.jit(h.idsva_so)(q1, qd1, qdd1)        # 4 x (1, NV, NV, NV)
idsva_np  = h_np.idsva_so(np.asarray(q1), np.asarray(qd1), np.asarray(qdd1))

names = ('d2tau_dq', 'd2tau_dqd', 'd2tau_cross', 'dM_dq')
worst = 0.0
for name, A, R in zip(names, idsva_jax, idsva_np):
    A = np.asarray(A[0]); R = np.asarray(R[0])
    scale = max(1.0, float(np.max(np.abs(R))))
    e = float(np.max(np.abs(A - R))) / scale
    print(f'  {name:11s} shape={A.shape}  max|err|/scale = {e:.2e}')
    worst = max(worst, e)
assert worst < 1e-3, worst
print('idsva_so (JAX) matches the numpy handle.')

JAX tour done: `jax.jit` over GRiD's dynamics, the native batch axis (the
vmap-equivalent), GRiD's **analytic** first/second-order gradient kernels
(jitted, FD-validated), and the note that autodiff *through* the FFI call is a
torch-backend feature. Every number was cross-checked, so a green *Run All*
validates the JAX surface, not just "no exception."